# Trip Duration Variability

## Definition

The 80th/20th Percentile ratio of durations between specified points on a vehicle’s trip. Reported for the start of the TSP corridor to the end of the TSP corridor and between each Timepoint along the TSP corridor.

## Reporting Grain

Per Timepoint, Service Pattern, Day Part (not yet implemented), and Reporting Period (currently ad hoc for a week).

## Source Data

GTFS-Schedule and Trip Updates

In [ ]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import datetime
import shapely

### Parameters

In [ ]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

In [ ]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

### Implementation

In [ ]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [ ]:
def make_linestring(x: str) -> shapely.geometry.LineString:
    # shapely errors if the array contains only one point
    if len(x) > 1:
        # each point in the array is wkt
        # so convert them to shapely points via list comprehension
        as_wkt = [shapely.wkt.loads(i) for i in x]
        return shapely.geometry.LineString(as_wkt)
    return shapely.geometry.LineString()

# Get the target route and feeds, one row per service date and shape
matching_shapes_df = pd.read_sql_query(
    env.get_template("shapes_routes.sql").render(CONSTANT_MACROS),
    engine
)
# convert dates from bigquery from datetime to date
matching_shapes_df["service_date"] = matching_shapes_df["service_date"].dt.date
# drop dates where the feed ran but the target route did not
matching_shapes_df = matching_shapes_df.loc[matching_shapes_df["shape_id"].notna()]

# the feed in effect can change part way through the window, so keep the mapping
# from each service date to the feed that was active on it
feed_dates = matching_shapes_df[["service_date", "feed_key"]].drop_duplicates()
feed_keys = feed_dates["feed_key"].drop_duplicates().to_list()

# collapse to one row per shape, with trips summed over all the selected dates
# natural key for shapes is feed key + shape id
matching_shapes_df["ct"] = matching_shapes_df.groupby("shape_id")["ct"].transform("sum")
matching_shapes_df = (
    matching_shapes_df
    .sort_values("service_date", ascending=True)
    .drop_duplicates(subset=["shape_id", "feed_key"], keep="last")
    .rename(columns={"service_date": "last_seen_date"})
    .sort_values("ct", ascending=False)
    .reset_index(drop=True)
)
matching_shapes_df = matching_shapes_df.loc[
    matching_shapes_df["ct"] >= MIN_TRIPS
]

# Get the geometry separately, reading as few date partitions as possible - a shape
# does not change within a feed version, so take the last date each shape was active
geometry_dates = matching_shapes_df["last_seen_date"].unique()
shape_geometry = pd.read_sql_query(
    env.get_template("shape_geometry.sql").render(
        {
            "GEOMETRY_DATES": geometry_dates,
            "FEED_KEYS": feed_keys,
            "SHAPE_IDS": matching_shapes_df["shape_id"].to_list(),
        }
    ),
    engine
)
matching_shapes_df = matching_shapes_df.merge(
    shape_geometry.drop_duplicates(subset=["shape_id", "feed_key"])[
        ["shape_id", "feed_key", "pt_array"]
    ],
    on=["shape_id", "feed_key"],
    how="left",
    validate="one_to_one",
).dropna(subset=["pt_array"]) #TODO: figure out why there are sometimes na geometries
display(matching_shapes_df)
#assert matching_shapes_df["pt_array"].notna().all()

matching_shapes_geom = gpd.GeoSeries(
    matching_shapes_df["pt_array"].map(make_linestring), crs=4326
)
matching_shapes = gpd.GeoDataFrame(
    matching_shapes_df.drop(["pt_array"], axis=1),
    geometry=matching_shapes_geom,
)
display(feed_dates)
matching_shapes

In [ ]:
# Get the schedule feed entry for every feed that was active during the window
schedule_feed_entries = pd.read_sql_query(
    env.get_template("schedule_feed_entry.sql").render(
        {"FEED_KEYS": feed_keys}
    ),
    engine
)
assert schedule_feed_entries.index.size == len(feed_keys)
# attach each service date to the feed that was active on it, so the stop times
# query can pair them up rather than mixing feed versions together
feed_dates = feed_dates.merge(schedule_feed_entries, on="feed_key", validate="many_to_one")
feed_dates

In [ ]:
# Get timepoint stops and schedule rt stop times for every selected date
schedule_rt_stop_times = pd.read_sql_query(
    env.get_template("schedule_rt_stop_times.sql").render(
        {
            "FEED_DATES": feed_dates.to_dict("records"),
            "FEED_KEYS": feed_keys,
            "FEED_VALID_FROMS": feed_dates["_valid_from"].drop_duplicates().to_list(),
            "SCHEDULE_BASE64_URLS": feed_dates["base64_url"].drop_duplicates().to_list(),
            "SHAPE_IDS": matching_shapes.shape_id.to_list(),
            **CONSTANT_MACROS
        }
    ),
    engine
)

# Get times between stops
# trip_id is only unique within a service date, so group by both. shape_id is
# redundant for well formed data but stops a trip that somehow appears under two
# shapes from bridging a travel time across them
schedule_rt_stop_times["prior_actual_departure_pacific"] = (
    schedule_rt_stop_times.groupby(
        ["service_date", "shape_id", "feed_key", "trip_id"]
    )["actual_departure_pacific"].shift()
)
schedule_rt_stop_times["time_difference"] = (
    schedule_rt_stop_times["actual_arrival_pacific"] 
    - schedule_rt_stop_times["prior_actual_departure_pacific"]
)    
schedule_rt_stop_times["time_difference_cleaned"] = schedule_rt_stop_times["time_difference"].where(
    schedule_rt_stop_times["time_difference"] >= datetime.timedelta(0),
    pd.NaT
)
schedule_rt_stop_times["time_difference_seconds"] = (
    schedule_rt_stop_times["time_difference_cleaned"].dt.seconds
)
schedule_rt_stop_times.sort_values(["shape_id", "feed_key"])

In [ ]:
# calculate the variance metric
# trips from every selected date are pooled into a single sample per segment
p80_time_difference = schedule_rt_stop_times.groupby(
    ["shape_id", "feed_key", "stop_sequence"]
)["time_difference_seconds"].quantile(0.8)
p80_time_difference.name = "p80_travel_time_seconds"
p20_time_difference = schedule_rt_stop_times.groupby(
    ["shape_id", "feed_key", "stop_sequence"]
)["time_difference_seconds"].quantile(0.2)
p20_time_difference.name = "p20_travel_time_seconds"
time_difference_variance_metric = p80_time_difference / p20_time_difference
time_difference_variance_metric.name = "travel_time_variability"
n_trips_per_timepoint = schedule_rt_stop_times.groupby(["shape_id", "feed_key", "stop_sequence"])["trip_id"].count()
n_trips_per_timepoint.name = "n_rt_trips"
# a shape only covers the dates its feed was active on, so record how many of the
# selected dates actually fed each segment
n_dates_per_timepoint = schedule_rt_stop_times.groupby(
    ["shape_id", "feed_key", "stop_sequence"]
)["service_date"].nunique()
n_dates_per_timepoint.name = "n_service_dates"
output_time_difference_variance = pd.concat(
    [
        time_difference_variance_metric,
        p80_time_difference,
        p20_time_difference,
        n_trips_per_timepoint,
        n_dates_per_timepoint,
    ],
    axis=1
).reset_index()

### Results

In [ ]:
# Total rt stop arrivals per day
# note fewer stop arrivals on start and end of the week
schedule_rt_stop_times["actual_arrival_pacific"].dt.date.value_counts()

In [ ]:
# Validity dates of different shapes, keyed by shape_id + feed_key
shape_date_validity = (
    schedule_rt_stop_times
    .groupby(["shape_id", "feed_key"])
    .agg(
        first_service_date=("service_date", "min"),
        last_service_date=("service_date", "max"),
        n_service_dates=("service_date", "nunique"),
        service_dates=(
            "service_date",
            lambda dates: ", ".join(map(str, sorted(dates.unique()))),
        ),
    )
)
# trip_id is only unique within a service date
shape_date_validity["n_rt_trips"] = (
    schedule_rt_stop_times[["shape_id", "feed_key", "service_date", "trip_id"]]
    .drop_duplicates()
    .groupby("shape_id")
    .size()
)
# where this is False the shape only ran for part of the window, so its map and its
# metrics are pooled over fewer days than the other shapes'
shape_date_validity["covers_all_target_dates"] = (
    shape_date_validity["n_service_dates"] == len(TARGET_DATES)
)
shape_date_validity = shape_date_validity.reset_index()
shape_date_validity

In [ ]:
# Time variance metric per segment
output_time_difference_variance

In [ ]:
# Maps of each segment

# Get a gdf of stops
timepoints = schedule_rt_stop_times[["shape_id", "feed_key", "stop_sequence", "stop_id", "pt_geom"]].drop_duplicates()
timepoints_gdf = gpd.GeoDataFrame(
    timepoints.drop(["pt_geom"], axis=1), 
    geometry=gpd.GeoSeries(timepoints["pt_geom"].map(shapely.wkt.loads), crs=4326)
)

for shape_id, feed_key, shape_geom in matching_shapes[["shape_id", "feed_key", "geometry"]].itertuples(index=False):
    timepoints_for_shape = timepoints_gdf.loc[
        (timepoints_gdf.shape_id == shape_id)
        & (timepoints_gdf.feed_key == feed_key)
    ].copy()
    timepoints_for_shape["projected_distance"] = timepoints_for_shape.geometry.map(
        lambda point_geom: shape_geom.project(point_geom)
    )
    timepoints_for_shape["prior_projected_distance"] = timepoints_for_shape["projected_distance"].shift()
    timepoints_for_shape["segment_geometry"] = timepoints_for_shape[["projected_distance", "prior_projected_distance"]].apply(
        lambda row: (
            shapely.ops.substring(shape_geom, row["projected_distance"], row["prior_projected_distance"])
            if not row.isna().any()
            else None
        ),
        axis=1
    )
    segment_geometry_with_values = (
        timepoints_for_shape
        .set_geometry("segment_geometry")
        [
            ["shape_id", "feed_key", "stop_sequence", "stop_id", "segment_geometry"]
        ]
        .merge(
            output_time_difference_variance,
            on=["shape_id", "feed_key", "stop_sequence"],
            how="left"
        )
    )
    segment_geometry_with_values = segment_geometry_with_values.loc[
        segment_geometry_with_values.stop_sequence != 2, #TODO: there seems to be an issue where this segment overlaps the entire route, so I've manually removed it
    ]
    segment_geometry_with_values.set_crs(4326).explore(
        "travel_time_variability",
        tiles="CartoDB positron"
    ).save(f"{shape_id}_{feed_key}.html")
